# Feature Engineering 
**Purpose:** Transform master dataset into ML-ready features

Steps:
1. Load master dataset
2. Lag features
3. Price change features
4. Conflict event flag
5. Risk level label
6. Interaction features
7. Outlier flagging
8. Scaling
9. Save final ML-ready dataset

In [1]:

#& import libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [52]:

#* load dataset
master = pd.read_csv('C:/Users/anasq/OneDrive/Desktop/PetroAnalyst/Dataset/ML_Ready_Data/master_dataset.csv')

#^ sort date values for safety
master = master.sort_values('date').reset_index(drop=True)
 
#? inspection
print('Shape:', master.shape)
print('Columns:', list(master.columns))
print('\nHead:')
master.head(3)

Shape: (195, 9)
Columns: ['Unnamed: 0', 'date', 'brent_price', 'wti_price', 'usd_rate', 'GPR', 'avg_lpg_price', 'avg_petrol', 'avg_diesel']

Head:


,Unnamed: 0,date,brent_price,wti_price,usd_rate,GPR,avg_lpg_price,avg_petrol,avg_diesel
0,0,2010/01/01,76.17,78.33,45.8944,91.58,353.4,50.89,40.04
1,1,2010/02/01,73.75,76.39,46.2732,80.73,353.4,50.35,37.31
2,2,2010/03/01,78.83,81.20,45.4509,74.12,358.4,50.89,40.04


# Step 1: Lag features (monthly)

In [3]:

#~ crude oil lag
#? takes 1-3 months to reflect in lpg price

master['brent_lag1'] = master['brent_price'].shift(1)
master['brent_lag3'] = master['brent_price'].shift(3)
master['brent_lag6'] = master['brent_price'].shift(6)

master['wti_lag1'] = master['wti_price'].shift(1)
master['wti_lag3'] = master['wti_price'].shift(3)

#! GPR lag (takes time to hit price)
master['gpr_lag1'] = master['GPR'].shift(1)
master['gpr_lag3'] = master['GPR'].shift(3)

#^ INR/USD lag (reflect on import price and delay)
master['usd_rate_lag1'] = master['usd_rate'].shift(1)
master['usd_rate_lag3'] = master['usd_rate'].shift(3)

#todo lpg price lag (past price decide future price)
master['lpg_price_lag1'] = master['avg_lpg_price'].shift(1)
master['lpg_price_lag3'] = master['avg_lpg_price'].shift(3)

#& petrol/ diesel price lag (depends on crude oil import)
master['petrol_price_lag1'] = master['avg_petrol'].shift(1)
master['petrol_price_lag3'] = master['avg_petrol'].shift(3)

master['diesel_price_lag1'] = master['avg_diesel'].shift(1)
master['diesel_price_lag3'] = master['avg_diesel'].shift(3)

print('Lag features added!\n')
print('new shape: ', master.shape)
print('head: \n', master.head())

Lag features added!

new shape:  (195, 24)
head: 
    Unnamed: 0        date  brent_price  wti_price  usd_rate    GPR  \
0           0  2010/01/01        76.17      78.33   45.8944  91.58   
1           1  2010/02/01        73.75      76.39   46.2732  80.73   
2           2  2010/03/01        78.83      81.20   45.4509  74.12   
3           3  2010/04/01        84.82      84.29   44.4440  88.76   
4           4  2010/05/01        75.95      73.74   45.7690  88.96   

   avg_lpg_price  avg_petrol  avg_diesel  brent_lag1  ...  gpr_lag1  gpr_lag3  \
0          353.4       50.89       40.04         NaN  ...       NaN       NaN   
1          353.4       50.35       37.31       76.17  ...     91.58       NaN   
2          358.4       50.89       40.04       73.75  ...     80.73       NaN   
3          363.4       50.89       40.04       78.83  ...     74.12     91.58   
4          368.4       51.44       40.50       84.82  ...     88.76     80.73   

   usd_rate_lag1  usd_rate_lag3  lpg_pric

# Step 2: Rolling averages

In [4]:

#* 3-month rolling average of crude, USD rate and GPR
master['brent_roll3'] = master['brent_price'].rolling(window=3).mean()
master['gpr_roll3']   = master['GPR'].rolling(window=3).mean()
master['inr_roll3']   = master['usd_rate'].rolling(window=3).mean()

#^ 6-month rolling average (brent + GPR index)
master['brent_roll6'] = master['brent_price'].rolling(window=6).mean()
master['gpr_roll6']   = master['GPR'].rolling(window=6).mean()

print('Rolling average features added.')
print('New shape:', master.shape)

Rolling average features added.
New shape: (195, 29)


# Step 3: price change feature (monthly)

In [5]:

#& % price changes 
master['brent_pct_change'] = master['brent_price'].pct_change() * 100
master['wti_pct_change']   = master['wti_price'].pct_change() * 100
master['gpr_pct_change']   = master['GPR'].pct_change() * 100
master['inr_pct_change']   = master['usd_rate'].pct_change() * 100

master['lpg_pct_change']   = master['avg_lpg_price'].pct_change() * 100
master['petrol_pct_change'] = master['avg_petrol'].pct_change() * 100
master['diesel_pct_change'] = master['avg_diesel'].pct_change() * 100

print('Price change features added.')
print('New shape:', master.shape)

Price change features added.
New shape: (195, 36)


# Step 4: Conflict event flag

In [6]:

#! global conflict dates
conflict_dates = {
    '2011/02/01': 'Arab Spring (Libya, Syria, Egypt unrest)',
    '2011/03/01': 'Arab Spring (Libya, Syria, Egypt unrest)',
    '2011/04/01': 'Arab Spring (Libya, Syria, Egypt unrest)',
    '2012/01/01': 'Iran sanctions imposed by US/EU',
    '2012/02/01': 'Iran sanctions imposed by US/EU',
    '2012/07/01': 'Iran sanctions imposed by US/EU',
    '2014/06/01': 'ISIS surge in Iraq',
    '2014/07/01': 'ISIS surge in Iraq',
    '2014/08/01': 'ISIS surge in Iraq',
    '2015/03/01': 'Yemen war start (Houthi-Saudi conflict)',
    '2015/04/01': 'Yemen war start (Houthi-Saudi conflict)',
    '2018/05/01': 'US exits Iran nuclear deal (JCPOA)',
    '2018/06/01': 'US exits Iran nuclear deal (JCPOA)',
    '2019/05/01': 'US-Iran Gulf tensions, tanker attacks',
    '2019/06/01': 'US-Iran Gulf tensions, tanker attacks',
    '2019/09/01': 'US-Iran Gulf tensions, tanker attacks',
    '2020/01/01': 'Soleimani assassination',
    '2022/02/01': 'Russia-Ukraine war',
    '2022/03/01': 'Russia-Ukraine war',
    '2022/04/01': 'Russia-Ukraine war',
    '2023/10/01': 'Israel-Palestine war',
    '2023/11/01': 'Israel-Palestine war',
    '2023/12/01': 'Israel-Palestine war',
    '2024/01/01': 'Houthi Red Sea attacks',
    '2024/02/01': 'Houthi Red Sea attacks',
    '2024/03/01': 'Houthi Red Sea attacks',
    '2024/04/01': 'Iran-Israel direct strikes',
    '2024/05/01': 'Iran-Israel direct strikes',
    '2026/01/01': 'Iran-USA 2026 conflict',
    '2026/02/01': 'Iran-USA 2026 conflict',
    '2026/03/01': 'Iran-USA 2026 conflict'
}


#? check in masters dataframe
master['conflict_event'] = master['date'].isin(conflict_dates).astype(int)

print('Conflict events flagged:', master['conflict_event'].sum(), 'months')

Conflict events flagged: 31 months


In [7]:

#^ values of parameters during conflict months
print('\nConflict months in dataset: with values\n\n')
print(master[master['conflict_event']==1].assign(event=master['date'].map(conflict_dates))[['date', 'event', 'GPR','brent_price','avg_lpg_price', 'avg_petrol', 'avg_diesel']].to_string())


Conflict months in dataset: with values


           date                                     event     GPR  brent_price  avg_lpg_price  avg_petrol  avg_diesel
13   2011/02/01  Arab Spring (Libya, Syria, Egypt unrest)   90.01       103.72          453.4       61.66       39.89
14   2011/03/01  Arab Spring (Libya, Syria, Egypt unrest)  136.89       114.64          533.4       61.38       39.89
15   2011/04/01  Arab Spring (Libya, Syria, Egypt unrest)   91.02       123.26          533.4       61.38       39.89
24   2012/01/01           Iran sanctions imposed by US/EU   79.63       110.69          880.4       68.82       47.18
25   2012/02/01           Iran sanctions imposed by US/EU   92.45       119.33          880.4       68.82       47.18
30   2012/07/01           Iran sanctions imposed by US/EU   82.81       102.62          922.9       71.76       49.30
53   2014/06/01                        ISIS surge in Iraq   98.77       111.80         1244.4       74.74       61.73
54   2014/07/

In [8]:

#todo percent change of parameters during conflict period
print('\nConflict months in dataset: with percent changes\n\n')
print(master[master['conflict_event']==1].assign(event=master['date'].map(conflict_dates))[['date','event', 'gpr_pct_change','brent_pct_change','wti_pct_change', 'inr_pct_change', 'lpg_pct_change', 'petrol_pct_change', 'diesel_pct_change']].to_string())


Conflict months in dataset: with percent changes


           date                                     event  gpr_pct_change  brent_pct_change  wti_pct_change  inr_pct_change  lpg_pct_change  petrol_pct_change  diesel_pct_change
13   2011/02/01  Arab Spring (Libya, Syria, Egypt unrest)       13.334173          7.459594       -0.661658        0.009917        9.675859           0.000000           0.000000
14   2011/03/01  Arab Spring (Libya, Syria, Egypt unrest)       52.083102         10.528346       16.121021       -1.025132       17.644464          -0.454103           0.000000
15   2011/04/01  Arab Spring (Libya, Syria, Egypt unrest)      -33.508657          7.519191        6.484542       -1.365489        0.000000           0.000000           0.000000
24   2012/01/01           Iran sanctions imposed by US/EU       -6.690884          2.614258        1.734984       -2.636191        0.000000           0.000000           0.000000
25   2012/02/01           Iran sanctions imposed by US/EU 

# Step 5: Risk level label

In [ ]:

#! Risk level according to gpr value
def assign_risk(gpr):
    if(gpr < 100): return 0 # low
    elif(gpr < 150): return 1 # medium
    elif(gpr < 200): return 2 # high
    else: return 3 # critical

master['risk_level'] = master['GPR'].apply(assign_risk)

#~ risk level distribution
risk_map = {0 : 'low', 1 : 'medium', 2 : 'high', 3 : 'critical'}
dist = master['risk_level'].map(risk_map).value_counts()
print('Risk dirtribution : \n')
print(dist, '\n\n')

print('Percentage: \n\n')
print((dist / len(master) * 100).round(1))

Risk dirtribution : 

risk_level
low         107
medium       74
high         10
critical      4
Name: count, dtype: int64 


Percentage: 


risk_level
low         54.9
medium      37.9
high         5.1
critical     2.1
Name: count, dtype: float64


# Step 6: Interaction features

In [15]:

#? 1. Crude oil cost in INR per barrel
master['crude_inr_impact'] = master['brent_price'] * master['usd_rate']

#^ 2. geopolitical pressure on crude
master['conflict_crude_index'] = (master['GPR'] / 100) * master['brent_price']

#* 3. Brent-WTI spread (widen during geopolitical events)
master['brent_wti_spread'] = master['brent_price'] - master['wti_price']

#! 4. INR stress during conflict — how much rupee weakens when GPR is high
master['inr_gpr_stress'] = master['usd_rate'] * (master['GPR'] / 100)

#~ 5. Crude oil price in INR terms (per avg lpg price)
master['crude_inr_normalized'] = (master['brent_price'] * master['usd_rate']) / master['avg_lpg_price']

#todo 6. LPG Price momentum — is LPG rising or falling in 3 months
master['lpg_momentum'] = master['avg_lpg_price'].diff(3) / 3 # +ve : rise, -ve : fall

#& 7. Conflict intensity score — combines GPR spike + crude spike
master['conflict_intensity'] = (
    (master['GPR'] - master['GPR'].mean()) / master['GPR'].std() +
    (master['brent_price'] - master['brent_price'].mean()) / master['brent_price'].std()
)

# 7. Outlier flagging

In [41]:

#* flag outliers based on quartile values (for GPR only)
def flag_outlier_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.0 * IQR
    upper = Q3 + 1.0 * IQR
    return ((series < lower) | (series > upper)).astype(int)

#& flag out;iers based on % change (for all other features)
def flag_outlier_pct(series, threshold=8.0):
    pct_change = series.pct_change() * 100
    return (pct_change.abs() > threshold).astype(int)


#todo detect outliers in parameters
master['gpr_outlier'] = flag_outlier_iqr(master['GPR'])
master['brent_outlier'] = flag_outlier_pct(master['brent_price'], threshold=18.0)
master['lpg_price_outlier'] = flag_outlier_pct(master['avg_lpg_price'], threshold=10.0)
master['petrol_price_outlier'] = flag_outlier_pct(master['avg_petrol'], threshold=5.0)
master['diesel_price_outlier'] = flag_outlier_pct(master['avg_diesel'])
master['usd_rate_outlier'] = flag_outlier_pct(master['usd_rate'], threshold=3.0)

print('Outlier flags added (1 = outlier, 0 = normal)')

print('Brent outliers:', master['brent_outlier'].sum())
print('GPR outliers:  ', master['gpr_outlier'].sum())
print('LPG outliers:  ', master['lpg_price_outlier'].sum())
print('Petrol outliers:  ', master['petrol_price_outlier'].sum())
print('Diesel outliers:  ', master['diesel_price_outlier'].sum())
print('INR outliers:  ', master['usd_rate_outlier'].sum())

print('\nOutlier months (GPR):')
print(master[master['gpr_outlier']==1][['date','GPR','brent_price','avg_lpg_price', 'avg_petrol', 'avg_diesel', 'usd_rate']])

Outlier flags added (1 = outlier, 0 = normal)
Brent outliers: 13
GPR outliers:   12
LPG outliers:   14
Petrol outliers:   14
Diesel outliers:   7
INR outliers:   14

Outlier months (GPR):
           date     GPR  brent_price  avg_lpg_price  avg_petrol  avg_diesel  \
145  2022/02/01  216.16        97.13          902.9      100.26       88.04   
146  2022/03/01  318.95       117.25          952.9      100.26       88.04   
147  2022/04/01  191.14       104.58         1002.9      100.26       88.04   
165  2023/10/01  197.89        90.60          911.7      103.26       91.04   
166  2023/11/01  156.70        82.94          911.7      103.26       91.04   
168  2024/01/01  160.37        80.12          911.7      105.26       92.04   
171  2024/04/01  163.95        89.94          811.7      105.26       92.04   
182  2025/03/01  173.91        72.73          811.7      110.26       97.04   
184  2025/05/01  165.66        64.45          861.7      110.26       97.04   
185  2025/06/01  222.3

# Step 8: clean data after adding new features 

In [44]:

#? remove first 6 records (contains missing values)
print('Shape before dropna:', master.shape)
print('Total NaN values:', master.isnull().sum().sum())

master = master.dropna().reset_index(drop=True)
print('Shape after dropna:', master.shape)
print('NaN values remaining:', master.isnull().sum().sum())

#^ round all numeric columns to 2 decimal places
numeric_cols = master.select_dtypes(include='number').columns
master[numeric_cols] = master[numeric_cols].round(2)

Shape before dropna: (189, 51)
Total NaN values: 0
Shape after dropna: (189, 51)
NaN values remaining: 0


In [ ]:

#! dataset preview
master.head(10)

,Unnamed: 0,date,brent_price,wti_price,usd_rate,GPR,avg_lpg_price,avg_petrol,avg_diesel,brent_lag1,...,inr_gpr_stress,crude_inr_normalized,lpg_momentum,conflict_intensity,brent_outlier,gpr_outlier,lpg_price_outlier,usd_rate_outlier,petrol_price_outlier,diesel_price_outlier
0,6,2010/07/01,75.58,76.32,46.76,79.38,378.4,54.45,42.32,74.76,...,37.12,9.34,5.00,-0.81,0,0,0,0,0,0
1,7,2010/08/01,77.04,76.60,46.46,80.99,383.4,54.55,39.88,75.58,...,37.63,9.34,5.00,-0.70,0,0,0,0,0,0
2,8,2010/09/01,77.84,75.24,45.87,71.17,388.4,54.77,39.98,77.04,...,32.65,9.19,5.00,-0.94,0,0,0,0,0,0
3,9,2010/10/01,82.67,81.89,44.35,65.99,393.4,55.47,39.89,77.84,...,29.27,9.32,5.00,-0.89,0,0,0,1,0,0
4,10,2010/11/01,85.28,84.25,44.93,94.71,398.4,55.78,39.89,82.67,...,42.55,9.62,5.00,0.03,0,0,0,0,0,0
5,11,2010/12/01,91.45,89.15,45.10,97.20,403.4,59.08,39.89,85.28,...,43.84,10.22,5.00,0.36,0,0,0,0,1,0
6,12,2011/01/01,96.52,89.17,45.38,79.42,413.4,61.66,39.89,91.45,...,36.04,10.59,6.67,0.07,0,0,0,0,0,0
7,13,2011/02/01,103.72,88.58,45.38,90.01,453.4,61.66,39.89,96.52,...,40.85,10.38,18.33,0.67,0,0,0,0,0,0
8,14,2011/03/01,114.64,102.86,44.91,136.89,533.4,61.38,39.89,103.72,...,61.48,9.65,43.33,2.44,0,0,1,0,0,0
9,15,2011/04/01,123.26,109.53,44.30,91.02,533.4,61.38,39.89,114.64,...,40.32,10.24,40.00,1.51,0,0,0,0,0,0


In [46]:
master 

,Unnamed: 0,date,brent_price,wti_price,usd_rate,GPR,avg_lpg_price,avg_petrol,avg_diesel,brent_lag1,...,inr_gpr_stress,crude_inr_normalized,lpg_momentum,conflict_intensity,brent_outlier,gpr_outlier,lpg_price_outlier,usd_rate_outlier,petrol_price_outlier,diesel_price_outlier
0,6,2010/07/01,75.58,76.32,46.76,79.38,378.4,54.45,42.32,74.76,...,37.12,9.34,5.0,-0.81,0,0,0,0,0,0
1,7,2010/08/01,77.04,76.60,46.46,80.99,383.4,54.55,39.88,75.58,...,37.63,9.34,5.0,-0.70,0,0,0,0,0,0
2,8,2010/09/01,77.84,75.24,45.87,71.17,388.4,54.77,39.98,77.04,...,32.65,9.19,5.0,-0.94,0,0,0,0,0,0
3,9,2010/10/01,82.67,81.89,44.35,65.99,393.4,55.47,39.89,77.84,...,29.27,9.32,5.0,-0.89,0,0,0,1,0,0
4,10,2010/11/01,85.28,84.25,44.93,94.71,398.4,55.78,39.89,82.67,...,42.55,9.62,5.0,0.03,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,190,2025/11/01,63.80,60.06,88.84,104.34,861.7,110.26,97.04,64.54,...,92.70,6.58,0.0,-0.60,0,0,0,0,0,0
185,191,2025/12/01,62.54,57.97,90.04,131.39,861.7,110.26,97.04,63.80,...,118.30,6.53,0.0,0.11,0,0,0,0,0,0
186,192,2026/01/01,66.60,60.04,90.90,167.80,861.7,107.26,94.04,62.54,...,152.53,7.03,0.0,1.31,0,1,0,0,0,0
187,193,2026/02/01,70.89,64.51,90.75,121.62,861.7,107.26,94.04,66.60,...,110.37,7.47,0.0,0.19,0,0,0,0,0,0


# Step 9: Create scaled version for Machine Learning

In [49]:

#^ colms to not scale
dont_scale = [
    'date',
    'risk_level',      
    'conflict_event',   
    'brent_outlier',    
    'gpr_outlier',      
    'lpg_outlier',      
    'inr_outlier',      
]

#todo colms to scale
scale_cols = [c for c in master.columns if c not in dont_scale]

scaler = MinMaxScaler()
master_scaled = master.copy()
master_scaled[scale_cols] = scaler.fit_transform(master[scale_cols])

print('Columns scaled:', scale_cols)
print('\nScaled data sample (all values should be 0-1):')
print(master_scaled[scale_cols[:5]].describe().round(3))

Columns scaled: ['Unnamed: 0', 'brent_price', 'wti_price', 'usd_rate', 'GPR', 'avg_lpg_price', 'avg_petrol', 'avg_diesel', 'brent_lag1', 'brent_lag3', 'brent_lag6', 'wti_lag1', 'wti_lag3', 'gpr_lag1', 'gpr_lag3', 'usd_rate_lag1', 'usd_rate_lag3', 'lpg_price_lag1', 'lpg_price_lag3', 'petrol_price_lag1', 'petrol_price_lag3', 'diesel_price_lag1', 'diesel_price_lag3', 'brent_roll3', 'gpr_roll3', 'inr_roll3', 'brent_roll6', 'gpr_roll6', 'brent_pct_change', 'wti_pct_change', 'gpr_pct_change', 'inr_pct_change', 'lpg_pct_change', 'petrol_pct_change', 'diesel_pct_change', 'crude_inr_impact', 'conflict_crude_index', 'brent_wti_spread', 'inr_gpr_stress', 'crude_inr_normalized', 'lpg_momentum', 'conflict_intensity', 'lpg_price_outlier', 'usd_rate_outlier', 'petrol_price_outlier', 'diesel_price_outlier']

Scaled data sample (all values should be 0-1):
       Unnamed: 0  brent_price  wti_price  usd_rate      GPR
count     189.000      189.000    189.000   189.000  189.000
mean        0.500        0.

# Step 10: Final summary

In [50]:
print('='*55)
print('   FEATURE ENGINEERING COMPLETE')
print('='*55)
print(f'Total rows:    {master.shape[0]}')
print(f'Total columns: {master.shape[1]}')
print(f'Date range:    {master["date"].iloc[0]} → {master["date"].iloc[-1]}')
print(f'Null values:   {master.isnull().sum().sum()}')
print(f'Conflict months flagged: {master["conflict_event"].sum()}')
print(f'\nRisk Level Distribution:')
risk_map = {0:"Low", 1:"Medium", 2:"High", 3:"Critical"}
print(master['risk_level'].map(risk_map).value_counts())
print(f'\nAll Columns:')
for i, col in enumerate(master.columns, 1):
    print(f'  {i:2}. {col}')

   FEATURE ENGINEERING COMPLETE
Total rows:    189
Total columns: 51
Date range:    2010/07/01 → 2026/03/01
Null values:   0
Conflict months flagged: 31

Risk Level Distribution:
risk_level
Low         101
Medium       74
High         10
Critical      4
Name: count, dtype: int64

All Columns:
   1. Unnamed: 0
   2. date
   3. brent_price
   4. wti_price
   5. usd_rate
   6. GPR
   7. avg_lpg_price
   8. avg_petrol
   9. avg_diesel
  10. brent_lag1
  11. brent_lag3
  12. brent_lag6
  13. wti_lag1
  14. wti_lag3
  15. gpr_lag1
  16. gpr_lag3
  17. usd_rate_lag1
  18. usd_rate_lag3
  19. lpg_price_lag1
  20. lpg_price_lag3
  21. petrol_price_lag1
  22. petrol_price_lag3
  23. diesel_price_lag1
  24. diesel_price_lag3
  25. brent_roll3
  26. gpr_roll3
  27. inr_roll3
  28. brent_roll6
  29. gpr_roll6
  30. brent_pct_change
  31. wti_pct_change
  32. gpr_pct_change
  33. inr_pct_change
  34. lpg_pct_change
  35. petrol_pct_change
  36. diesel_pct_change
  37. conflict_event
  38. risk_level

# Step 11: Save both files

In [51]:
master.to_csv('EDA_data.csv', index=False)
master_scaled.to_csv('ML_ready_data.csv')

print('\nFiles saved:')
print('  EDA_data.csv  -> use for EDA & visualization')
print('  ML_ready_data.csv    -> use for ML algorithms')


Files saved:
  EDA_data.csv  -> use for EDA & visualization
  ML_ready_data.csv    -> use for ML algorithms
